In [ ]:
%pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Load the dataset
q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(q1_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum())

df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])




In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df

In [ ]:
# Task 5: Write your code here:
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black', color="pink")

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

# Taget is balanced

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):


  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model = RandomForestRegressor()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  mae_scores.append(mean_absolute_error(y_test, y_pred))


mae_scores = np.array(mae_scores)
print(f"MAE:  {mae_scores.mean()}")

In [ ]:
# Task 1: Write your code here:
feature_cols = X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, alpha=0.7, color='green', edgecolor="black")
plt.title('Predicted Delivery Time Histogram')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostClassifier
mae_scores_ensemble = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model = RandomForestRegressor(random_state=42)
    cat_model = CatBoostRegressor()
    rf_model.fit(X_train, y_train)
    cat_model.fit(X_train, y_train)

    rf_pred = rf_model.predict(X_test)
    cat_pred = cat_model.predict(X_test)


    average_predictions = (rf_pred + cat_pred) / 2
    ensemble_mae = mean_absolute_error(y_test, average_predictions)
    mae_scores_ensemble.append(ensemble_mae)

average_ensemble_mae = sum(mae_scores_ensemble) / len(mae_scores_ensemble)
print(f"Average MAE for Ensemble model for all the folds are: {average_ensemble_mae}")
